<p align="center">
  <img src="https://img.shields.io/badge/BINX%20TECH-AI%20%26%20ML%20INTERNSHIP-0B1F3A?style=for-the-badge" alt="BinX Tech"/>
</p>

<h1 align="center">🌐 Week 9 · Day 2</h1>
<h3 align="center">Serving the Model with FastAPI</h3>

<p align="center">
  <img src="https://img.shields.io/badge/PHASE-3%20CAPSTONE-0B1F3A?style=for-the-badge" alt="Phase 3"/>
  <img src="https://img.shields.io/badge/SPRINT-4%20OF%204-1B4F8C?style=for-the-badge" alt="Sprint 4"/>
  <img src="https://img.shields.io/badge/DAY-2%20OF%205-0096C7?style=for-the-badge&logoColor=black" alt="Day 2 of 5"/>
  <img src="https://img.shields.io/badge/TOPIC-REST%20API-0077B6?style=for-the-badge" alt="REST API"/>
  <img src="https://img.shields.io/badge/CAPSTONE-CARDIAC%20MONITORING-03045E?style=for-the-badge" alt="Cardiac Monitoring Capstone"/>
</p>

<p align="center"><i>"Serving a model with FastAPI means exposing a URL that accepts input data in a request and returns the model's prediction in the response."</i></p>

---

## 📖 The Story So Far

Day 1 ended with two files sitting on disk — `model.joblib` and `preprocessor.joblib` — plus the discipline to trust
them (a verified round-trip prediction) and reproduce them (a pinned `requirements.txt`, an MLflow-logged run).
Today those two files stop being passive artifacts and become the brain behind a real, callable web service: a
FastAPI application with a `/predict` endpoint that any other program — a website, a mobile app, tomorrow's
Streamlit dashboard — can call over HTTP.

## 🎯 Learning Objectives

| | Objective |
|---|---|
| 🏗️ | Build a FastAPI app that loads a serialized model and preprocessing |
| 📮 | Create a `/predict` POST endpoint that validates input with Pydantic |
| 🧪 | Test the endpoint locally using FastAPI's automatic documentation |


## 1&nbsp;·&nbsp;What FastAPI Does

FastAPI is a modern Python framework for building REST APIs — web endpoints that other programs can call. Serving a
model with FastAPI means exposing a URL that accepts input data in a request and returns the model's prediction in
the response. This is how models are served in production: the model becomes a service any application can query,
instead of something that only runs inside one person's notebook.

## 2&nbsp;·&nbsp;Recap: What We're Loading

Today's app loads exactly what Day 1 produced — nothing is retrained or reimplemented here.

In [1]:
import os
import pandas as pd

# Day 2 is self-contained: bring in Day 1's frozen artifacts unchanged.
for f in ["model.joblib", "preprocessor.joblib", "requirements.txt"]:
    assert os.path.exists(f), f"Missing {f} -- copy Day 1's artifacts into this folder first."
    print(f"{f:<20} {os.path.getsize(f):>8,} bytes")

# Reload the same data used to train the shipped model, purely to read off its feature schema below.
df = pd.read_csv("heart.csv")
X = df.drop(columns=["HeartDisease"])

model.joblib            1,007 bytes
preprocessor.joblib     4,118 bytes
requirements.txt          285 bytes


## 3&nbsp;·&nbsp;Defining the Input Schema

The capstone's model expects the same 11 features it was trained on. A FastAPI app defines routes (URL paths) as
functions, and Pydantic's `BaseModel` defines and validates exactly what shape of data the `/predict` route accepts
— matching the project's real features, not a generic placeholder schema.

In [2]:
feature_dtypes = X.dtypes
print(feature_dtypes)

Age                 int64
Sex                   str
ChestPainType         str
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG            str
MaxHR               int64
ExerciseAngina        str
Oldpeak           float64
ST_Slope              str
dtype: object


Reading straight from the trained pipeline's own feature list (rather than retyping it by hand) is what
guarantees the API's schema can never drift out of sync with what the model actually expects.

## 4&nbsp;·&nbsp;Building the FastAPI App

The cell below writes `main.py` to disk — a real, standalone file. This is deliberate: an API is meant to run as its
own process (`uvicorn main:app`), not live only inside a notebook cell. Writing it out with `%%writefile` keeps the
actual deployable file and this notebook's walkthrough of it in sync.

In [3]:
main_py_lines = [
    "# Week 9 Day 2 -- Cardiac Monitoring prediction API.",
    "# Loads Day 1's serialized model + preprocessing and serves a /predict endpoint.",
    "from fastapi import FastAPI",
    "from pydantic import BaseModel, Field",
    "from typing import Literal",
    "import pandas as pd",
    "import joblib",
    "",
    "app = FastAPI(",
    '    title="Cardiac Monitoring Prediction API",',
    '    description="BinX Tech AI & ML Internship -- Phase 3 Capstone (Sprint 4)",',
    '    version="1.0.0",',
    ")",
    "",
    'model = joblib.load("model.joblib")',
    'preprocessor = joblib.load("preprocessor.joblib")',
    "",
    "",
    "# One patient's vitals -- matches heart.csv's 11 input features exactly.",
    "class PatientData(BaseModel):",
    '    Age: int = Field(..., ge=0, le=120, description="Age in years")',
    '    Sex: Literal["M", "F"]',
    '    ChestPainType: Literal["ATA", "NAP", "ASY", "TA"]',
    '    RestingBP: float = Field(..., ge=0, description="Resting blood pressure (mm Hg)")',
    '    Cholesterol: float = Field(..., ge=0, description="Serum cholesterol (mg/dl)")',
    '    FastingBS: Literal[0, 1] = Field(..., description="1 if fasting blood sugar > 120 mg/dl, else 0")',
    '    RestingECG: Literal["Normal", "ST", "LVH"]',
    '    MaxHR: float = Field(..., ge=0, description="Maximum heart rate achieved")',
    '    ExerciseAngina: Literal["Y", "N"]',
    '    Oldpeak: float = Field(..., description="ST depression induced by exercise")',
    '    ST_Slope: Literal["Up", "Flat", "Down"]',
    "",
    "    class Config:",
    "        json_schema_extra = {",
    '            "example": {',
    '                "Age": 54, "Sex": "M", "ChestPainType": "ASY", "RestingBP": 130,',
    '                "Cholesterol": 246, "FastingBS": 0, "RestingECG": "Normal",',
    '                "MaxHR": 150, "ExerciseAngina": "N", "Oldpeak": 1.0, "ST_Slope": "Flat",',
    "            }",
    "        }",
    "",
    "",
    "class PredictionResponse(BaseModel):",
    "    prediction: int",
    '    label: Literal["Disease", "No Disease"]',
    "    probability: float",
    "",
    "",
    '@app.get("/")',
    "def root():",
    '    return {"status": "ok", "message": "Cardiac Monitoring Prediction API -- see /docs"}',
    "",
    "",
    '@app.post("/predict", response_model=PredictionResponse)',
    "def predict(data: PatientData):",
    "    row = pd.DataFrame([data.model_dump()])",
    "    X_pre = preprocessor.transform(row)",
    "    pred = int(model.predict(X_pre)[0])",
    "    proba = float(model.predict_proba(X_pre)[0, 1])",
    "    return PredictionResponse(",
    "        prediction=pred,",
    '        label="Disease" if pred == 1 else "No Disease",',
    "        probability=round(proba, 4),",
    "    )",
    "",
]
main_py = "\n".join(main_py_lines)

with open("main.py", "w") as f:
    f.write(main_py)

print(main_py)

# Week 9 Day 2 -- Cardiac Monitoring prediction API.
# Loads Day 1's serialized model + preprocessing and serves a /predict endpoint.
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal
import pandas as pd
import joblib

app = FastAPI(
    title="Cardiac Monitoring Prediction API",
    description="BinX Tech AI & ML Internship -- Phase 3 Capstone (Sprint 4)",
    version="1.0.0",
)

model = joblib.load("model.joblib")
preprocessor = joblib.load("preprocessor.joblib")


# One patient's vitals -- matches heart.csv's 11 input features exactly.
class PatientData(BaseModel):
    Age: int = Field(..., ge=0, le=120, description="Age in years")
    Sex: Literal["M", "F"]
    ChestPainType: Literal["ATA", "NAP", "ASY", "TA"]
    RestingBP: float = Field(..., ge=0, description="Resting blood pressure (mm Hg)")
    Cholesterol: float = Field(..., ge=0, description="Serum cholesterol (mg/dl)")
    FastingBS: Literal[0, 1] = Field(..., description="1 if fast

## 5&nbsp;·&nbsp;Pydantic Validation

The `PatientData` class above does more than describe fields — it **validates** every incoming request against
those declared types automatically. If a caller sends text where a number is expected, omits a required field, or
sends a category outside the allowed `Literal` values (e.g. `"Sex": "male"` instead of `"M"`), FastAPI rejects the
request with a clear `422` error instead of the model silently receiving malformed input. This input validation is
essential for a robust service — never trust that incoming data is well-formed.

## 6&nbsp;·&nbsp;Testing the API

Running the server locally (`uvicorn main:app --reload`) exposes interactive documentation at `/docs`, where every
endpoint can be tested in the browser without writing a client. Inside this notebook, FastAPI's `TestClient` gives
the same real request → validation → preprocessing → model → response chain, without needing a separately running
server process.

In [4]:
from fastapi.testclient import TestClient
import importlib, sys

# Import the app we just wrote to main.py (fresh import, in case of an earlier run).
sys.path.insert(0, ".")
if "main" in sys.modules:
    importlib.reload(sys.modules["main"])
import main as api_module

client = TestClient(api_module.app)

print("GET / ->", client.get("/").json())

GET / -> {'status': 'ok', 'message': 'Cardiac Monitoring Prediction API -- see /docs'}


### Valid request

A real patient row (the same one used for Day 1's round-trip check), sent exactly as JSON the way a website or
mobile app would send it.

In [5]:
valid_payload = {
    "Age": 54, "Sex": "M", "ChestPainType": "ASY", "RestingBP": 130,
    "Cholesterol": 246, "FastingBS": 0, "RestingECG": "Normal",
    "MaxHR": 150, "ExerciseAngina": "N", "Oldpeak": 1.0, "ST_Slope": "Flat",
}

resp = client.post("/predict", json=valid_payload)
print("Status code:", resp.status_code)
print("Response body:", resp.json())

Status code: 200
Response body: {'prediction': 1, 'label': 'Disease', 'probability': 0.8214}


### Invalid request — Pydantic rejects it before the model ever sees it

Sending a string where a number is expected (`"Age": "fifty-four"`) and an out-of-vocabulary category
(`"Sex": "unknown"`) demonstrates the validation working, not just being declared.

In [6]:
invalid_payload = {
    "Age": "fifty-four",          # should be a number
    "Sex": "unknown",              # not one of "M" / "F"
    "ChestPainType": "ASY", "RestingBP": 130, "Cholesterol": 246,
    "FastingBS": 0, "RestingECG": "Normal", "MaxHR": 150,
    "ExerciseAngina": "N", "Oldpeak": 1.0, "ST_Slope": "Flat",
}

resp = client.post("/predict", json=invalid_payload)
print("Status code:", resp.status_code)
for err in resp.json()["detail"]:
    print(f"  - {'.'.join(str(p) for p in err['loc'])}: {err['msg']}")

Status code: 422
  - body.Age: Input should be a valid integer, unable to parse string as an integer
  - body.Sex: Input should be 'M' or 'F'


> [!NOTE]
> **The deployment payoff of Day 1:** the API above applies the *exact same* preprocessing as training — loaded from
> `preprocessor.joblib`, not re-implemented. That single line (`preprocessor.transform(row)`) is what makes today's
> service trustworthy: there is no second, hand-written copy of the scaling/encoding logic that could ever drift out
> of sync with what the model was actually trained on.

## 7&nbsp;·&nbsp;Running the Server for Real

Outside this notebook, the same `main.py` runs as an actual live service:

```bash
uvicorn main:app --reload
# Then open http://127.0.0.1:8000/docs to test /predict interactively in the browser
```

`--reload` restarts the server automatically on code changes during development. The `/docs` page is generated
automatically from the `PatientData` schema above — including the example payload — so anyone on the team can try
the API without reading a line of code.

## 🧪 Hands-On Lab Recap: Building a Prediction API

| Step | Done |
|---|---|
| 1. Write a FastAPI app that loads the serialized model and preprocessing from Day 1 | ✅ Section 4 |
| 2. Define an input schema with Pydantic matching the project's features | ✅ Section 4 |
| 3. Implement a `/predict` endpoint that preprocesses input and returns the prediction as JSON | ✅ Section 4 |
| 4. Run the server locally and test `/predict` through the `/docs` interface with several inputs | ✅ Sections 6–7 |
| 5. Confirm invalid input is rejected cleanly by Pydantic, documented in the notebook | ✅ Section 6 |

## 📦 Artifacts Produced Today

- `main.py` — the real, deployable FastAPI application (not just notebook code)
- A verified request → validation → preprocessing → model → response chain, tested for both valid and invalid input

**Tomorrow (Day 3):** this same API becomes the backend for an interactive Streamlit dashboard, so a
non-technical user can get a prediction without ever touching `/docs` or JSON.

## 🧰 Tools Used

![FastAPI](https://img.shields.io/badge/FastAPI-0B1F3A?style=flat-square)
![Pydantic](https://img.shields.io/badge/Pydantic-1B4F8C?style=flat-square)
![Uvicorn](https://img.shields.io/badge/Uvicorn-0096C7?style=flat-square)
![joblib](https://img.shields.io/badge/joblib-0B1F3A?style=flat-square)
![Jupyter](https://img.shields.io/badge/Jupyter%20%2F%20Colab-0B1F3A?style=flat-square)
![Git](https://img.shields.io/badge/Git%20%26%20GitHub-0B1F3A?style=flat-square)

---

<p align="center"><sub>Week 9 · Sprint 4 · Day 2 of 5 → <b>Day 3: Interactive Streamlit Dashboard</b></sub></p>
